In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Fecom Inc. — Phân tích dữ liệu thương mại điện tử
**Sử dụng Apache Spark DataFrame**

## Khởi tạo SparkSession

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Fecom_Ecommerce_Analysis") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.0.2


---
## Câu 1: Đọc dữ liệu từ các file CSV (tự suy ra kiểu dữ liệu)

In [3]:
# Đường dẫn tới thư mục chứa dữ liệu
DATA_PATH = "/content/drive/MyDrive/DATA-SCIENCE-SUBJECT/DS200/lab/lab-4/LAB-4/data/"

# Hàm tiện ích để đọc CSV với separator ';'
def read_csv(filename):
    return spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("sep", ";") \
        .csv(DATA_PATH + filename)

# Đọc tất cả các file
orders_df       = read_csv("Orders.csv")
customers_df    = read_csv("Customer_List.csv")
order_items_df  = read_csv("Order_Items.csv")
products_df     = read_csv("Products.csv")
reviews_df      = read_csv("Order_Reviews.csv")

# Kiểm tra schema
print("=== Orders ===")
orders_df.printSchema()

print("=== Customers ===")
customers_df.printSchema()

print("=== Order Items ===")
order_items_df.printSchema()

print("=== Products ===")
products_df.printSchema()

print("=== Order Reviews ===")
reviews_df.printSchema()

=== Orders ===
root
 |-- Order_ID: string (nullable = true)
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Purchase_Timestamp: timestamp (nullable = true)
 |-- Order_Approved_At: timestamp (nullable = true)
 |-- Order_Delivered_Carrier_Date: timestamp (nullable = true)
 |-- Order_Delivered_Customer_Date: timestamp (nullable = true)
 |-- Order_Estimated_Delivery_Date: timestamp (nullable = true)

=== Customers ===
root
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Subscriber_ID: string (nullable = true)
 |-- Subscribe_Date: date (nullable = true)
 |-- First_Order_Date: date (nullable = true)
 |-- Customer_Postal_Code: string (nullable = true)
 |-- Customer_City: string (nullable = true)
 |-- Customer_Country: string (nullable = true)
 |-- Customer_Country_Code: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)

=== Order Items ===
root
 |-- Order_ID: string (nullable = true)
 |-

---
## Câu 2: Thống kê tổng số đơn hàng, khách hàng và người bán

In [4]:
total_orders   = orders_df.select("Order_ID").distinct().count()
total_customers = customers_df.select("Subscriber_ID").distinct().count()
total_sellers  = order_items_df.select("Seller_ID").distinct().count()

print(f"Tổng số đơn hàng  : {total_orders:,}")
print(f"Số khách hàng duy nhất: {total_customers:,}")
print(f"Số người bán duy nhất : {total_sellers:,}")

Tổng số đơn hàng  : 99,441
Số khách hàng duy nhất: 99,382
Số người bán duy nhất : 3,095


---
## Câu 3: Phân tích số lượng đơn hàng theo quốc gia (giảm dần)

In [5]:
orders_by_country = (
    orders_df
    .join(customers_df, on="Customer_Trx_ID", how="left")
    .groupBy("Customer_Country", "Customer_Country_Code")
    .agg(F.count("Order_ID").alias("Total_Orders"))
    .orderBy(F.col("Total_Orders").desc())
)

orders_by_country.show(30, truncate=False)

+----------------+---------------------+------------+
|Customer_Country|Customer_Country_Code|Total_Orders|
+----------------+---------------------+------------+
|Germany         |DE                   |41754       |
|France          |FR                   |12848       |
|Netherlands     |NL                   |11629       |
|Belgium         |BE                   |5464        |
|Austria         |AT                   |5043        |
|Switzerland     |CH                   |3640        |
|United Kingdom  |GB                   |3382        |
|Poland          |PL                   |2139        |
|Czechia         |CZ                   |2034        |
|Italy           |IT                   |2025        |
|Spain           |ES                   |1651        |
|Portugal        |PT                   |1336        |
|Sweden          |SE                   |975         |
|Denmark         |DK                   |905         |
|Serbia          |RS                   |746         |
|Norway          |NO        

---
## Câu 4: Phân tích số lượng đơn hàng theo năm và tháng
> Hiển thị theo năm tăng dần, tháng giảm dần

In [6]:
orders_by_ym = (
    orders_df
    .filter(F.col("Order_Purchase_Timestamp").isNotNull())
    .withColumn("Year",  F.year("Order_Purchase_Timestamp"))
    .withColumn("Month", F.month("Order_Purchase_Timestamp"))
    .groupBy("Year", "Month")
    .agg(F.count("Order_ID").alias("Total_Orders"))
    .orderBy(
        F.col("Year").asc(),
        F.col("Month").desc()
    )
)

orders_by_ym.show(50, truncate=False)

+----+-----+------------+
|Year|Month|Total_Orders|
+----+-----+------------+
|2022|12   |1           |
|2022|10   |324         |
|2022|9    |4           |
|2023|12   |5673        |
|2023|11   |7544        |
|2023|10   |4631        |
|2023|9    |4285        |
|2023|8    |4331        |
|2023|7    |4026        |
|2023|6    |3245        |
|2023|5    |3700        |
|2023|4    |2404        |
|2023|3    |2682        |
|2023|2    |1780        |
|2023|1    |800         |
|2024|10   |4           |
|2024|9    |16          |
|2024|8    |6512        |
|2024|7    |6292        |
|2024|6    |6167        |
|2024|5    |6873        |
|2024|4    |6939        |
|2024|3    |7211        |
|2024|2    |6728        |
|2024|1    |7269        |
+----+-----+------------+



---
## Câu 5: Thống kê điểm đánh giá trung bình và số lượng theo từng mức
> Xử lý giá trị ngoại lệ và NULL trong cột `Review_Score`

In [8]:
# Kiểm tra dữ liệu trước khi xử lý
print("--- Phân phối Review_Score trước khi làm sạch ---")
reviews_df.groupBy("Review_Score").count().orderBy("Review_Score").show()

print(f"Số giá trị NULL: {reviews_df.filter(F.col('Review_Score').isNull()).count()}")

# Chỉ giữ các điểm hợp lệ từ 1 đến 5
reviews_clean = reviews_df.withColumn(
    "Review_Score_Clean",
    F.expr("try_cast(Review_Score as int)")
).filter(
    F.col("Review_Score_Clean").isNotNull() &
    F.col("Review_Score_Clean").between(1, 5)
).drop("Review_Score").withColumnRenamed("Review_Score_Clean", "Review_Score")

print(f"\nSố bản ghi hợp lệ sau làm sạch: {reviews_clean.count():,}")
print(f"Số bản ghi bị loại           : {reviews_df.count() - reviews_clean.count():,}")

# Điểm trung bình tổng thể
avg_score = reviews_clean.agg(F.avg("Review_Score").alias("Avg_Score")).collect()[0]["Avg_Score"]
if avg_score is not None:
    print(f"\nĐiểm đánh giá trung bình tổng thể: {avg_score:.2f}")
else:
    print("\nKhông có điểm đánh giá hợp lệ (Vui lòng kiểm tra lại cấu trúc file CSV)")

# Thống kê theo từng mức điểm
print("\n--- Thống kê theo từng mức điểm ---")
review_stats = (
    reviews_clean
    .groupBy("Review_Score")
    .agg(
        F.count("Review_ID").alias("Count"),
        F.round(F.count("Review_ID") / reviews_clean.count() * 100, 2).alias("Percentage_%")
    )
    .orderBy("Review_Score")
)
review_stats.show()

--- Phân phối Review_Score trước khi làm sạch ---
+----------------+-----+
|    Review_Score|count|
+----------------+-----+
|            NULL|   45|
|               1|11424|
|               2| 3151|
|2024-04-07 04:19|    1|
|2024-07-05 11:11|    1|
|               3| 8179|
|               4|19141|
|               5|57328|
+----------------+-----+

Số giá trị NULL: 45

Số bản ghi hợp lệ sau làm sạch: 99,223
Số bản ghi bị loại           : 47

Điểm đánh giá trung bình tổng thể: 4.09

--- Thống kê theo từng mức điểm ---
+------------+-----+------------+
|Review_Score|Count|Percentage_%|
+------------+-----+------------+
|           1|11424|       11.51|
|           2| 3151|        3.18|
|           3| 8179|        8.24|
|           4|19141|       19.29|
|           5|57328|       57.78|
+------------+-----+------------+



---
## Câu 6: Doanh thu năm 2024 theo danh mục sản phẩm
> Doanh thu = Giá sản phẩm (Price) + Phí vận chuyển (Freight_Value)

In [9]:
# Lọc đơn hàng năm 2024
orders_2024 = orders_df.filter(
    F.year("Order_Purchase_Timestamp") == 2024
).select("Order_ID")

revenue_2024 = (
    orders_2024
    .join(order_items_df, on="Order_ID", how="inner")
    .join(products_df,    on="Product_ID", how="left")
    .withColumn("Revenue", F.col("Price") + F.col("Freight_Value"))
    .groupBy("Product_Category_Name")
    .agg(
        F.round(F.sum("Revenue"), 2).alias("Total_Revenue"),
        F.count("Order_ID").alias("Total_Items_Sold")
    )
    .orderBy(F.col("Total_Revenue").desc())
)

print("=== Doanh thu năm 2024 theo danh mục sản phẩm ===")
revenue_2024.show(30, truncate=False)

# Tổng doanh thu
total_rev = revenue_2024.agg(F.sum("Total_Revenue")).collect()[0][0]
print(f"Tổng doanh thu năm 2024: {total_rev:,.2f}")

=== Doanh thu năm 2024 theo danh mục sản phẩm ===
+-------------------------------------+-------------+----------------+
|Product_Category_Name                |Total_Revenue|Total_Items_Sold|
+-------------------------------------+-------------+----------------+
|Health_Beauty                        |885191.12    |5951            |
|Watches_Gifts                        |771986.75    |3703            |
|Bed_Bath_Table                       |650794.7     |5884            |
|Sports_Leisure                       |621999.34    |4527            |
|Computers_Accessories                |594771.04    |4708            |
|Housewares                           |491576.96    |4046            |
|Furniture_Decor                      |476466.13    |4118            |
|Auto                                 |404210.57    |2619            |
|Baby                                 |299052.56    |1776            |
|Cool_Stuff                           |273910.05    |1473            |
|Garden_Tools              

---
## Câu 7: Sản phẩm bán nhiều nhất & điểm đánh giá trung bình từng sản phẩm

In [10]:
# Số lượng bán ra và điểm đánh giá trung bình theo Product_ID
product_sales = (
    order_items_df
    .groupBy("Product_ID")
    .agg(F.count("Order_Item_ID").alias("Units_Sold"))
)

product_reviews = (
    order_items_df
    .join(
        reviews_clean.select("Order_ID", "Review_Score"),
        on="Order_ID",
        how="left"
    )
    .groupBy("Product_ID")
    .agg(F.round(F.avg("Review_Score"), 2).alias("Avg_Review_Score"))
)

product_analysis = (
    product_sales
    .join(product_reviews, on="Product_ID", how="left")
    .join(products_df.select("Product_ID", "Product_Category_Name"), on="Product_ID", how="left")
    .orderBy(F.col("Units_Sold").desc())
)

print("=== Top 20 sản phẩm bán chạy nhất ===")
product_analysis.show(20, truncate=False)

# Sản phẩm bán nhiều nhất
top_product = product_analysis.first()
print(f"\nSản phẩm bán nhiều nhất:")
print(f"  Product_ID   : {top_product['Product_ID']}")
print(f"  Danh mục     : {top_product['Product_Category_Name']}")
print(f"  Số lượng bán : {top_product['Units_Sold']}")
print(f"  Điểm TB      : {top_product['Avg_Review_Score']}")

=== Top 20 sản phẩm bán chạy nhất ===
+--------------------------------+----------+----------------+---------------------+
|Product_ID                      |Units_Sold|Avg_Review_Score|Product_Category_Name|
+--------------------------------+----------+----------------+---------------------+
|aca2eb7d00ea1a7b8ebd4e68314663af|527       |4.02            |Furniture_Decor      |
|99a4788cb24856965c36a24e339b6058|488       |3.9             |Bed_Bath_Table       |
|422879e10f46682990de24d770e7f83d|484       |3.95            |Garden_Tools         |
|389d119b48cf3043d311335e499d9c6b|392       |4.12            |Garden_Tools         |
|368c6c730842d78016ad823897a372db|388       |3.92            |Garden_Tools         |
|53759a2ecddad2bb87a079a1f1519f73|373       |3.87            |Garden_Tools         |
|d1c427060a0f73f6b889a5c7c61f2ac4|343       |4.19            |Computers_Accessories|
|53b36df67ebb7c41585e8d54d6772e08|323       |4.19            |Watches_Gifts        |
|154e7e31ebfa092203795c972e

---
## Câu 8: Hiệu suất giao hàng
> Hiệu số giữa ngày giao hàng thực tế (`Order_Delivered_Carrier_Date`) và ngày dự kiến (`Shipping_Limit_Date`)

In [11]:
# Tính hiệu số ngày (dương = trễ, âm = sớm hơn hạn)
delivery_perf = (
    orders_df
    .filter(F.col("Order_Delivered_Carrier_Date").isNotNull())
    .join(
        order_items_df.select("Order_ID", "Shipping_Limit_Date"),
        on="Order_ID",
        how="inner"
    )
    .withColumn(
        "Delivery_Diff_Days",
        F.datediff(
            F.col("Order_Delivered_Carrier_Date"),
            F.col("Shipping_Limit_Date")
        )
    )
    .withColumn(
        "Delivery_Status",
        F.when(F.col("Delivery_Diff_Days") <= 0, "On-time / Early")
         .otherwise("Late")
    )
)

print("=== Mẫu dữ liệu hiệu suất giao hàng ===")
delivery_perf.select(
    "Order_ID",
    "Shipping_Limit_Date",
    "Order_Delivered_Carrier_Date",
    "Delivery_Diff_Days",
    "Delivery_Status"
).show(15, truncate=False)

# Tổng hợp thống kê
print("=== Thống kê hiệu suất giao hàng ===")
delivery_perf.groupBy("Delivery_Status") \
    .agg(
        F.count("Order_ID").alias("Order_Count"),
        F.round(F.avg("Delivery_Diff_Days"), 2).alias("Avg_Diff_Days"),
        F.min("Delivery_Diff_Days").alias("Min_Diff_Days"),
        F.max("Delivery_Diff_Days").alias("Max_Diff_Days")
    ) \
    .show(truncate=False)

# Tỷ lệ giao hàng đúng hạn
total = delivery_perf.count()
on_time = delivery_perf.filter(F.col("Delivery_Status") == "On-time / Early").count()
print(f"\nTỷ lệ giao hàng đúng hạn: {on_time}/{total} = {on_time/total*100:.1f}%")

=== Mẫu dữ liệu hiệu suất giao hàng ===
+--------------------------------+-------------------+----------------------------+------------------+---------------+
|Order_ID                        |Shipping_Limit_Date|Order_Delivered_Carrier_Date|Delivery_Diff_Days|Delivery_Status|
+--------------------------------+-------------------+----------------------------+------------------+---------------+
|00010242fe8c5a6d1ba2dd792cb16214|2023-09-19 09:45:00|2023-09-19 18:34:00         |0                 |On-time / Early|
|00018f77f2f0320c557190d7a144bdd3|2023-05-03 11:05:00|2023-05-04 14:35:00         |1                 |Late           |
|000229ec398224ef6ca0657da4fc703e|2024-01-18 14:48:00|2024-01-16 12:36:00         |-2                |On-time / Early|
|00024acbcdf0a6daa1e931b038114c75|2024-08-15 10:10:00|2024-08-10 13:28:00         |-5                |On-time / Early|
|00042b26cf59d7ce69dfabb4e55b4fd9|2023-02-13 13:57:00|2023-02-16 09:46:00         |3                 |Late           |
|00048cc

---
## Kết thúc

In [12]:
spark.stop()
print("SparkSession đã dừng.")

SparkSession đã dừng.
